In [0]:
import json
from pyspark.sql import functions as F

payload = spark.table("workspace.raw.weather_raw").select("payload").first()["payload"]
h = json.loads(payload)["hourly"]
rows = list(zip(h["time"], h["temperature_2m"], h["precipitation"]))

weather = (
    spark.createDataFrame(rows, ["hour_ts", "temperature_c", "precipitation_mm"])
         .withColumn("hour_ts", F.to_timestamp("hour_ts"))
)

weather.write.mode("overwrite").saveAsTable("workspace.clean.weather_hourly")

print(spark.table("workspace.clean.weather_hourly").count())
display(spark.table("workspace.clean.weather_hourly").limit(5))

1440


hour_ts,temperature_c,precipitation_mm
2016-01-01T00:00:00.000Z,3.1,0.0
2016-01-01T01:00:00.000Z,2.9,0.0
2016-01-01T02:00:00.000Z,3.0,0.0
2016-01-01T03:00:00.000Z,2.7,0.0
2016-01-01T04:00:00.000Z,2.9,0.0


In [0]:
from pyspark.sql import functions as F

t = spark.table("workspace.raw.taxi_trips")

reason = (
    F.when(F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"), "dropoff_not_after_pickup")
     .when(F.col("trip_distance") <= 0, "distance_le_0")
     .when(F.col("fare_amount") <= 0, "fare_le_0")
)
flagged = t.withColumn("reject_reason", reason)

rejected = flagged.filter(F.col("reject_reason").isNotNull())
good = flagged.filter(F.col("reject_reason").isNull()).drop("reject_reason")

trips_clean = (
    good.withColumn("pickup_hour", F.date_trunc("hour", "tpep_pickup_datetime"))
        .withColumn("trip_minutes",
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
)

rejected.write.mode("overwrite").saveAsTable("workspace.clean.trips_rejected")
trips_clean.write.mode("overwrite").saveAsTable("workspace.clean.trips_clean")

w = spark.table("workspace.clean.weather_hourly")
trips_weather = (
    spark.table("workspace.clean.trips_clean")
         .join(w, F.col("pickup_hour") == F.col("hour_ts"), "left")
         .drop("hour_ts")
)
trips_weather.write.mode("overwrite").saveAsTable("workspace.clean.trips_weather")

# data quality checks
n_raw = t.count()
n_clean = spark.table("workspace.clean.trips_clean").count()
n_rej = spark.table("workspace.clean.trips_rejected").count()
n_join = spark.table("workspace.clean.trips_weather").count()
n_nomatch = spark.table("workspace.clean.trips_weather").filter(F.col("temperature_c").isNull()).count()

print("raw:", n_raw, "| clean:", n_clean, "| rejected:", n_rej, "| joined:", n_join, "| no weather match:", n_nomatch)
assert n_raw == n_clean + n_rej, "rows lost between raw and clean"
assert n_join == n_clean, "join changed the row count"
assert n_nomatch == 0, "some trips have no weather"

display(spark.table("workspace.clean.trips_rejected").groupBy("reject_reason").count())

raw: 21932 | clean: 21847 | rejected: 85 | joined: 21847 | no weather match: 0


reject_reason,count
distance_le_0,75
fare_le_0,9
dropoff_not_after_pickup,1
